# 🎚️ 실시간 성격 조종 — Activation Steering

**가중치를 바꾸지 않고**, 추론하는 *순간* 활성화에 '방향'을 더해 — 실시간으로 성격·말투를 바꾼다.

> 거부 방향(#5)과 같은 원리(diff-of-means)지만, 여기선 **무해한 성격**(밝음↔차가움)으로 안전하게 배운다.

**비유**: 약을 먹이듯 — 그 순간만 "더 밝게", 끄면 원래대로. 학습(되돌리기 어려움)과 다르다.
**논문**: Contrastive Activation Addition (Rimsky 2024, 2312.06681) · **실행**: 무료 Colab(T4)

## 1단계 — 설치

In [ ]:
!pip -q install transformers accelerate torch

## 2단계 — 작은 모델 로드

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto").eval()
DEV = model.device

## 3단계 — 성격 '대비' 문장으로 방향 찾기

**밝은** 답 vs **차가운** 답의 활성화 차이 = '밝음 방향'. (거부 방향과 똑같은 diff-of-means)

In [ ]:
bright = ["정말 신나요!", "와, 최고예요! 도와드릴게요 :)", "기분 좋은 하루예요!", "당연하죠, 즐겁게 해봐요!",
          "멋져요! 응원할게요!", "함께라서 행복해요!", "좋아요, 신나게 시작해요!", "기대돼요, 화이팅!"]
cold   = ["그러든지요.", "관심 없어요.", "알아서 하세요.", "딱히 도와줄 생각 없어요.",
          "귀찮네요.", "그게 뭐 중요한가요.", "마음대로 하세요.", "별로 신경 안 써요."]

@torch.no_grad()
def mid_act(texts):
    acts=[]
    for t in texts:
        ids = tok.apply_chat_template([{"role":"assistant","content":t}], tokenize=True, return_tensors="pt").to(DEV)
        hs = model(ids, output_hidden_states=True).hidden_states
        acts.append(torch.stack(hs)[:, 0, -1, :].float().cpu())   # [L+1, hidden]
    return torch.stack(acts).mean(0)                                # [L+1, hidden]

mu_b, mu_c = mid_act(bright), mid_act(cold)
directions = (mu_b - mu_c); directions = directions / directions.norm(dim=-1, keepdim=True)
L = model.config.num_hidden_layers
best = int(0.6 * L)                                                 # 중간~60% 레이어가 성격을 잘 담음
bright_dir = directions[best + 1].to(DEV, dtype=model.dtype)        # hidden_states[best+1] = layers[best] 출력
print(f"✅ '밝음 방향' 추출 — 레이어 {best}/{L}")

## 4단계 — 주입 hook + 같은 질문에 성격 바꿔보기

In [ ]:
def make_hook(vec, alpha):
    def hook(m, i, o):
        h = o[0] if isinstance(o, tuple) else o
        h = h + alpha * vec
        return (h,)+o[1:] if isinstance(o, tuple) else h
    return hook

@torch.no_grad()
def chat(p):
    ids = tok.apply_chat_template([{"role":"user","content":p}], add_generation_prompt=True, return_tensors="pt").to(DEV)
    out = model.generate(ids, max_new_tokens=60, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

Q = "오늘 일이 많아서 힘들어."
print("🟡 기본:", chat(Q))
h = model.model.layers[best].register_forward_hook(make_hook(bright_dir, 8.0))
print("🟢 밝게(+):", chat(Q)); h.remove()
h = model.model.layers[best].register_forward_hook(make_hook(bright_dir, -8.0))
print("🔵 차갑게(−):", chat(Q)); h.remove()

## 5단계 — 세기(alpha) 조절

조종 강도를 키우면 성격이 더 진해진다. (너무 키우면 말이 깨짐 — 적당히)

In [ ]:
for a in [0, 4, 8, 12]:
    h = model.model.layers[best].register_forward_hook(make_hook(bright_dir, a))
    print(f"alpha={a:2d} →", chat("회의 끝났어.")[:80]); h.remove()

---
## 🎓 무슨 일이 일어난 건가

- 밝은/차가운 답의 활성화 **차이 벡터**를 구해, 추론 중 활성화에 **더하니 성격이 실시간으로** 바뀌었다.
- **가중치는 그대로** — hook을 떼면 즉시 원래대로. 학습과 달리 **켜고 끄는 조종**.
- 거부 방향(#5)과 **완전히 같은 원리** — 대상이 '거부' 대신 '성격'일 뿐. 방향을 알면 무엇이든 밀고 당길 수 있다.

> ⚖️ **윤리**: 이 노트북은 **무해한 성격(밝음↔차가움)** 으로 원리만 배운다. 안전장치를 푸는 용도가 아니다.

## 📚 레퍼런스
| 주제 | 논문 | arXiv |
| --- | --- | --- |
| 실시간 조종(CAA) | Contrastive Activation Addition (Rimsky) | 2312.06681 |
| 거부 방향(같은 원리) | Refusal Direction (Arditi) | 2406.11717 |